# LAB | Agents that speak MCP fluently

**Path:** one LangChain agent → three MCP servers → one grounded answer.

| Server | Source | What it provides |
|---|---|---|
| `filesystem` | [`@modelcontextprotocol/server-filesystem`](https://github.com/modelcontextprotocol/servers/tree/main/src/filesystem), restricted to `docs/` | tools: read / list / search docs |
| `git` | [`mcp-server-git`](https://github.com/modelcontextprotocol/servers/tree/main/src/git), pointed at a clone of [`langchain-ai/langchain-mcp-adapters`](https://github.com/langchain-ai/langchain-mcp-adapters) (`GIT_SOURCE_REPO`) | tools: log / show / diff history |
| `docs` | `docs_resource_server.py` (custom FastMCP) | **resources**: `docs://*.md` |

Flow: `INPUT query → DECIDE tool path → BUILD (connect, load tools, run agent) → VERIFY (trace + grounding check) → EXPLAIN (failure mode)`

## Step 1 — Setup: load environment and verify the OpenAI key

In [1]:
import os, sys, json, re
from pathlib import Path
from dotenv import load_dotenv

LAB = Path.cwd()  # run the notebook from the repo root
load_dotenv(LAB / ".env")

# Source for the git server: a clone of https://github.com/langchain-ai/langchain-mcp-adapters
GIT_REPO = Path(os.getenv("GIT_SOURCE_REPO", "")).expanduser()
assert (GIT_REPO / ".git").exists(), "Set GIT_SOURCE_REPO in .env to a clone of langchain-ai/langchain-mcp-adapters"

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found in .env"
print("OpenAI key loaded:", True, "| git source:", GIT_REPO)
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
print("Model:", MODEL)

OpenAI key loaded: True | git source: <GIT_SOURCE_REPO>
Model: gpt-4o-mini


## Step 2 — Connect to MCP servers with `MultiServerMCPClient`

The filesystem server is sandboxed to `docs/` on purpose: pointing it at the repo root would expose `.env`.

### Guardrail: tool-call interceptor

First run without this failed: the agent called `git_log(max_count=50)`, got **516k chars** back (dependabot commit bodies), and the request exceeded the model's 128k-token context. Second run (cap 10 commits + truncate at 8k) also failed: 10 commits were still 91k chars, so truncation cut off the target commit and the agent **looped**, calling tools again and again searching for it. Final guardrail: (a) compact `git_log` to one line per commit (hash, author, date, subject) so every commit stays visible, (b) cap at 20 commits, (c) truncate any other output above `MAX_TOOL_CHARS` with a visible marker so truncation itself is traceable, and (d) a `recursion_limit` on the agent run.

In [2]:
import re
from mcp.types import CallToolResult, TextContent

MAX_TOOL_CHARS = 8_000
MAX_GIT_LOG = 20

def compact_git_log(text):
    """Keep hash, author, date and the first message line of each commit."""
    lines = []
    for entry in re.split(r"\n(?=Commit: )", text):
        fields = dict(re.findall(r"^(Commit|Author|Date|Message): (.*)$", entry, flags=re.M))
        if "Commit" in fields:
            lines.append(f"{fields['Commit']} | {fields.get('Date', '')} | {fields.get('Author', '')} | {fields.get('Message', '')}")
    return "Commit history (compacted by guardrail: hash | date | author | subject):\n" + "\n".join(lines)

async def size_guardrail(request, handler):
    if request.name == "git_log":
        args = dict(request.args)
        args["max_count"] = min(int(args.get("max_count", MAX_GIT_LOG)), MAX_GIT_LOG)
        request = request.override(args=args)
    result = await handler(request)
    if isinstance(result, CallToolResult):
        new_content = []
        for block in result.content:
            if request.name == "git_log" and isinstance(block, TextContent):
                block = TextContent(type="text", text=compact_git_log(block.text))
            if isinstance(block, TextContent) and len(block.text) > MAX_TOOL_CHARS:
                omitted = len(block.text) - MAX_TOOL_CHARS
                block = TextContent(type="text", text=block.text[:MAX_TOOL_CHARS] + f"\n[TRUNCATED by guardrail: {omitted:,} chars omitted]")
            new_content.append(block)
        result = CallToolResult(content=new_content, isError=result.isError, structuredContent=result.structuredContent)
    return result

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", str(LAB / "docs")],
    },
    "git": {
        "transport": "stdio",
        "command": sys.executable,
        "args": ["-m", "mcp_server_git", "--repository", str(GIT_REPO)],
    },
    "docs": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [str(LAB / "docs_resource_server.py")],
    },
}, tool_interceptors=[size_guardrail])

# Test the connection: list raw tools and resources per server
for name in ["filesystem", "git", "docs"]:
    async with client.session(name) as session:
        tools = (await session.list_tools()).tools
        try:
            resources = [str(r.uri) for r in (await session.list_resources()).resources]
        except Exception as e:  # reference servers don't implement resources
            resources = f"not supported ({e})"
        print(f"{name}: {len(tools)} tools | resources: {resources}")

filesystem: 14 tools | resources: not supported (Method not found)


git: 12 tools | resources: not supported (Method not found)


docs: 0 tools | resources: ['docs://langchain_mcp_adapters_README.md']


## Step 3 — Load MCP tools as LangChain tools

In v0.3.x the method is `client.get_tools()` (the lab text says `get_langchain_tools()`, which does not exist). We keep **read-only tools only** as a guardrail — the agent cannot `write_file`, `git_commit` or `git_reset`.

In [4]:
READ_ONLY = {
    "read_text_file", "read_multiple_files", "list_directory", "directory_tree",
    "search_files", "get_file_info", "list_allowed_directories",
    "git_status", "git_log", "git_show", "git_diff", "git_branch",
}

all_tools = await client.get_tools()
tools = [t for t in all_tools if t.name in READ_ONLY]
print(f"Loaded {len(all_tools)} tools, kept {len(tools)} read-only:")
for t in tools:
    print(f"  - {t.name}: {t.description.splitlines()[0][:90]}")

Loaded 26 tools, kept 12 read-only:
  - read_text_file: Read the complete contents of a file from the file system as text. Handles various text en
  - read_multiple_files: Read the contents of multiple files simultaneously. This is more efficient than reading fi
  - list_directory: Get a detailed listing of all files and directories in a specified path. Results clearly d
  - directory_tree: Get a recursive tree view of files and directories as a JSON structure. Each entry include
  - search_files: Recursively search for files and directories matching a pattern. The patterns should be gl
  - get_file_info: Retrieve detailed metadata about a file or directory. Returns comprehensive information in
  - list_allowed_directories: Returns the list of directories that this server is allowed to access. Subdirectories with
  - git_status: Shows the working tree status
  - git_diff: Shows differences between branches or commits
  - git_log: Shows the commit logs
  - git_show: Shows the contents o

## Step 5 — Access MCP resources

`client.get_resources(server)` returns LangChain `Blob`s. We put a **catalog** of resources into the system prompt (URI + title), not the full text, so the agent still has to call a tool to quote evidence — that keeps the trace inspectable.

In [5]:
blobs = await client.get_resources("docs")
catalog = []
for b in blobs:
    text = b.as_string()
    title = next((l.lstrip("# ").strip() for l in text.splitlines() if l.startswith("#")), "")
    catalog.append(f"- {b.metadata['uri']} — {title} ({len(text):,} chars)")
    print(b.metadata["uri"], "|", b.mimetype, "|", len(text), "chars | first line:", text.splitlines()[0][:80])
RESOURCE_CATALOG = "\n".join(catalog)

docs://langchain_mcp_adapters_README.md | text/markdown | 11924 chars | first line: # LangChain MCP Adapters


## Step 4 / 6 — Create the MCP-enabled agent

In [6]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

SYSTEM_PROMPT = f"""You are a repository analyst. You answer ONLY from evidence returned by your MCP tools.

Capabilities:
- filesystem tools: read/search the documentation folder {LAB / 'docs'}
- git tools: inspect the commit history of the repository at {GIT_REPO}

Available documentation resources (catalog only — read the file with a tool before quoting it):
{RESOURCE_CATALOG}

Rules:
1. Call tools before answering. Never answer from memory.
2. Cite every claim with its source: (git_show <hash>) or (file: <name>).
3. If the tools do not return the evidence, say "Not found in sources" instead of guessing.
"""

llm = ChatOpenAI(model=MODEL, temperature=0)
agent = create_agent(llm, tools, system_prompt=SYSTEM_PROMPT)
print(SYSTEM_PROMPT)

You are a repository analyst. You answer ONLY from evidence returned by your MCP tools.

Capabilities:
- filesystem tools: read/search the documentation folder <LAB>/docs
- git tools: inspect the commit history of the repository at <GIT_SOURCE_REPO>

Available documentation resources (catalog only — read the file with a tool before quoting it):
- docs://langchain_mcp_adapters_README.md — LangChain MCP Adapters (11,924 chars)

Rules:
1. Call tools before answering. Never answer from memory.
2. Cite every claim with its source: (git_show <hash>) or (file: <name>).
3. If the tools do not return the evidence, say "Not found in sources" instead of guessing.



## Run one query

The query needs **both** tool servers: git history (the commit) and the docs (the README).

In [7]:
QUERY = (
    "Which commit capped the mcp dependency below 2.0.0, and what exactly changed in pyproject.toml? "
    "Also, according to the README, how do I connect to multiple MCP servers at once?"
)
RUN_CONFIG = {"recursion_limit": 25}
result = await agent.ainvoke({"messages": [{"role": "user", "content": QUERY}]}, config=RUN_CONFIG)
messages = result["messages"]
print(messages[-1].content)

The commit that capped the MCP dependency below 2.0.0 is identified by the hash **d4a1caa9fb76f8f5206e274c60abe8bd5a3a2599**. The changes made in the `pyproject.toml` file were as follows:

- The dependency line for `mcp` was changed from:
  ```plaintext
  "mcp>=1.24.0",
  ```
  to:
  ```plaintext
  "mcp>=1.24.0,<2.0.0",
  ```

This change was made to prevent import failures that occurred when new installations resolved to MCP SDK version 2.x, which removed necessary components used by the package (git_show d4a1caa9).

### Connecting to Multiple MCP Servers
According to the README, you can connect to multiple MCP servers using the `MultiServerMCPClient`. Here’s a brief overview of how to do it:

1. **Server Setup**: Define multiple servers in a dictionary format, specifying the command and transport for each server.
   
   Example:
   ```python
   client = MultiServerMCPClient(
       {
           "math": {
               "command": "python",
               "args": ["/path/to/math_serv

## VERIFY — Trace view: every tool call, its arguments, and the raw tool output

In [8]:
from langchain_core.messages import AIMessage, ToolMessage

def tool_text(m):
    c = m.content
    return c if isinstance(c, str) else "\n".join(p.get("text", "") for p in c if isinstance(p, dict))

def show_trace(messages, max_chars=1200):
    trace = []
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            for tc in m.tool_calls:
                print(f"\n>>> TOOL CALL  {tc['name']}({json.dumps(tc['args'])})")
                trace.append({"type": "call", "name": tc["name"], "args": tc["args"]})
        elif isinstance(m, ToolMessage):
            out = tool_text(m)
            print(f"<<< TOOL OUTPUT [{m.name}] ({len(out):,} chars)\n{out[:max_chars]}{' …[truncated]' if len(out) > max_chars else ''}")
            trace.append({"type": "output", "name": m.name, "content": out})
    print("\n=== FINAL ANSWER ===\n" + messages[-1].content)
    return trace

trace = show_trace(messages)


>>> TOOL CALL  git_log({"repo_path": "<GIT_SOURCE_REPO>", "max_count": 20})

>>> TOOL CALL  read_text_file({"path": "<LAB>/docs/langchain_mcp_adapters_README.md"})
<<< TOOL OUTPUT [git_log] (3,259 chars)
Commit history (compacted by guardrail: hash | date | author | subject):
52a4535f3eb4b98f386836e4d9b8c4cadf99afca | 2026-09-10 22:21:45-07:00 | Sydney Runkle | docs: announce MCP migration (#628)
cc0f2843fd5becf2cdb641c910533c3747c3b1ef | 2026-08-06 02:14:08-04:00 | Mason Daugherty | release: 0.3.2 (#604)
d4a1caa9fb76f8f5206e274c60abe8bd5a3a2599 | 2026-08-06 13:04:50+08:00 | 石岳峰 | fix(deps): cap mcp below 2.0.0 to prevent import failures (#590)
a060231535a54b20c1f635340b37df9b171edbb5 | 2026-08-05 01:27:20-07:00 | dependabot[bot] | build(deps): bump cryptography from 48.0.1 to 50.0.0 (#599)
c50072ae3cd9d68547474abf434c7b2d42d25caa | 2026-08-05 01:27:13-07:00 | dependabot[bot] | build(deps): bump cryptography from 48.0.1 to 50.0.0 in /examples/servers/streamable-http-stateless (#600)
e

## VERIFY — Automatic grounding check

A cheap check: every commit hash, every `code span`, and every line of a fenced code block in the answer must appear verbatim in some tool output. Anything flagged ✗ is a claim the trace does not support.

In [9]:
def grounding_check(answer, trace):
    evidence = "\n".join(t["content"] for t in trace if t["type"] == "output")
    # Lines inside ``` fences are checked one by one; inline `spans` and commit hashes as-is
    fenced = re.findall(r"```[^\n]*\n(.*?)```", answer, flags=re.S)
    prose = re.sub(r"```.*?```", "", answer, flags=re.S)
    claims = set(re.findall(r"\b[0-9a-f]{7,40}\b", prose)) | set(re.findall(r"`([^`\n]+)`", prose))
    claims |= {line.strip() for block in fenced for line in block.splitlines() if len(line.strip()) > 3}
    rows = [(c, c in evidence) for c in sorted(claims)]
    for c, ok in rows:
        print(("✓" if ok else "✗"), c)
    supported = sum(ok for _, ok in rows)
    print(f"\n{supported}/{len(rows)} checkable claims found verbatim in tool output")
    return rows

check = grounding_check(messages[-1].content, trace)

✓ "args": ["/path/to/math_server.py"],
✓ "command": "python",
✓ "math": {
✓ "mcp>=1.24.0",
✓ "mcp>=1.24.0,<2.0.0",
✓ "transport": "http",
✓ "transport": "stdio",
✓ "url": "http://localhost:8000/mcp",
✓ "weather": {
✓ MultiServerMCPClient
✓ agent = create_agent("openai:gpt-4.1", tools)
✓ client = MultiServerMCPClient(
✓ d4a1caa9
✓ d4a1caa9fb76f8f5206e274c60abe8bd5a3a2599
✓ mcp
✓ pyproject.toml
✓ tools = await client.get_tools()

17/17 checkable claims found verbatim in tool output


## EXPLAIN — Failure-case probe

Ask something the sources **cannot** answer. Does the agent say "Not found in sources", or does it produce a plausible but unsupported answer? Inspect the trace and the grounding check, then write your interpretation in `lab_proof.md`.

In [10]:
FAIL_QUERY = "On what date was version 0.4.0 of this library released, and who approved that release?"
from langgraph.errors import GraphRecursionError

try:
    fail_result = await agent.ainvoke({"messages": [{"role": "user", "content": FAIL_QUERY}]}, config=RUN_CONFIG)
    fail_trace = show_trace(fail_result["messages"], max_chars=600)
    fail_answer = fail_result["messages"][-1].content
    print()
    grounding_check(fail_answer, fail_trace)
except GraphRecursionError as e:
    # The agent kept calling tools without ever concluding "Not found in sources"
    fail_trace, fail_answer = [], f"NO ANSWER: {type(e).__name__} after {RUN_CONFIG['recursion_limit']} steps"
    print(fail_answer)

NO ANSWER: GraphRecursionError after 25 steps


## Save evidence for `lab_proof.md`

`MultiServerMCPClient` (v0.3.x) opens a fresh stdio session per call and closes it afterwards, so there is no long-lived connection to disconnect.

In [11]:
evidence = {
    "query": QUERY, "trace": trace, "final_answer": messages[-1].content, "grounding_check": check,
    "fail_query": FAIL_QUERY, "fail_trace": fail_trace, "fail_answer": fail_answer,
}
(LAB / "evidence.json").write_text(json.dumps(evidence, indent=2, ensure_ascii=False))
print("Saved", LAB / "evidence.json")

Saved <LAB>/evidence.json
